# **PROJECT PHASE II**
**Rachel MacMillan // rem388** <br>
**Valentina Fernández // jvf48**

## **Research Question:** <br>
**How do patterns of leisure activities, including television viewing, social media use, reading, and exercise, differ across age groups, and how are these differences associated with cognitive and mental health assessment outcomes?**

Research on prolonged passive media consumption (such as binge-watching or "doomscrolling") shows that these activities negatively affect cognitive function and mental-wellbeing. Alternatively, active leisure activities such as reading and exercise are frequently associated with positive cognitive and psychological outcomes. 

Determining how leisure activities affect cognitive function and mental-wellbeing typically requires controlled experimental studies, which are difficult to conduct and beyond the scope of this project. Instead, we will investigate how leisure activity patterns vary across age groups and whether those patterns are associated with differences in cognitive and mental health assessments. 

By combining datasets that measure leisure activities, demographic characteristics, and cognitive or mental health outcomes, we hope to identify meaningful relationships between lifestyle behaviors and well-being. In particular, we are interested in whether age groups characterized by greater participation in passive screen-based activities differ from groups that spend more time reading or exercising with respect to measures such as distractibility, forgetfulness, and overall mental health. While our analysis will not establish causation, it may reveal patterns that help motivate future research on how different forms of leisure activity relate to cognitive and psychological well-being.

## **Datasets:**

**[2025 American Time Use Survey: Activity Summary File](https://www.bls.gov/tus/data/datafiles-2025.htm):** <br>
The U.S. Census Bureau annually conducts the American Time Use Survey, which measures how American's divide their time among daily activities. These activities are organized into major categories such as personal care, eating and drinking, household activities, work, education, leisure and sports, and communication, with each category further divided into more detailed subcategories. 

ATUS participants are randomly selected from a subset of households that have completed their eighth month of interviews for the U.S. Census Population Survey. Participants are then interviewed once to collect time-use data of their previous day. Demographic information such as age, sex, martial status, occupation, income, and education are also recorded for each participant. 

ATUS provides several microdatasets regarding information on the respondant, activity, and activity summary. We will use ATUS's [Activity Summary File](https://drive.google.com/drive/folders/1Etczii3Ya7hL1ITElyc8QnaYF7_iJU_3?usp=sharing), which includes demographical and time-use data for each of the 6,145 participants. 


## **Data Cleaning & Descriptions:**

In [2]:
!pip install pandas numpy duckdb

In [3]:
import pandas as pd
import numpy as np
import duckdb

### **Cleaning American Time Use Dataset**

In [4]:
atus_summ = pd.read_csv("data/atusSummary/atussum_2025.dat")
atus_summ.head()

,TUCASEID,TUFINLWGT,TRYHHCHILD,TEAGE,TESEX,PEEDUCA,PTDTRACE,PEHSPNON,GTMETSTA,TELFS,...,t181401,t181501,t181599,t181601,t181801,t189999,t500101,t500103,t500105,t500106
0,20250101250011,3.193152e+07,-1,55,1,39,1,2,2,1,...,0,0,0,0,0,0,0,0,0,0
1,20250101250092,3.583641e+07,-1,58,2,41,1,2,1,5,...,0,0,0,0,0,0,0,0,0,0
2,20250101250283,2.607856e+07,-1,24,1,39,1,2,1,1,...,0,0,0,0,0,0,0,0,0,15
3,20250101250305,3.535833e+07,-1,25,2,44,7,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,20250101250594,5.520668e+06,-1,80,2,39,1,2,1,5,...,0,0,0,0,0,0,0,0,0,0


We used ATUS's [User Guide](https://www.bls.gov/tus/atususersguide.pdf) to determine which leisure activity each lexicode variable represented. 

**relevant atus_summ variables:** 
- `TUCASEID`: unique identifier for each participant
- `TEAGE`: age of participant
- `t010101`: minutes spent sleeping per day
- `t120303`: minutes spent watching TV per day
- `t120307`: minutes spent playing games (such as board games, videogames, or mobile games) per day
- `t120308`: minutes spent using computer for leisure (such as websurfing or using social media; **excludes gaming**) per day
- `t120309`: minutes spent doing arts and crafts as a hobby per day
- `t120312`: minutes spent reading for personal interest per day
- `t120313`: minutes spent writing for personal interest per day
- `t130101`: minutes spent exercising (working out or playing sports) per day

We will now create a new dataframe `activities` which includes the relevant variables and we will rename them appropriately. 

In [5]:
#activites shows the age of each participant and the average minutes they spent on each activity per day
activities = pd.DataFrame({
    "id" : atus_summ["TUCASEID"],
    "age" : atus_summ["TEAGE"],
    "sleeping" : atus_summ["t010101"],
    "watchingTV" : atus_summ["t120303"],
    "gaming" : atus_summ["t120307"],
    "onComputer" : atus_summ["t120308"],
    "artsCrafts" : atus_summ["t120309"],
    "reading" : atus_summ["t120312"],
    "writing" : atus_summ["t120313"],
    "exercising" : atus_summ["t130101"]
})
print("activites: the average minutes each participant spent on each activity per day")
activities.head(10)


activites: the average minutes each participant spent on each activity per day


,id,age,sleeping,watchingTV,gaming,onComputer,artsCrafts,reading,writing,exercising
0,20250101250011,55,540,30,0,0,0,0,0,0
1,20250101250092,58,725,270,0,0,0,0,0,0
2,20250101250283,24,390,83,0,0,0,0,0,0
3,20250101250305,25,580,0,0,0,0,135,0,0
4,20250101250594,80,840,290,0,0,0,0,0,0
5,20250101250600,67,660,0,0,0,0,0,0,0
6,20250101250630,62,615,180,0,0,0,0,0,0
7,20250101250640,72,690,265,0,0,0,30,0,0
8,20250101250657,78,565,60,0,0,0,0,0,0
9,20250101250816,46,850,0,0,0,0,0,0,0


As a sanity check, we will compare our statistics with [ATUS's Lastest Numbers](https://www.bls.gov/tus/latest-numbers.htm), specifically ATUS reports that American's spent on average 9.03 hours sleeping in 2025. 

In [6]:
avg_hours_sleeping = (activities["sleeping"].mean())/60
print("Average Hours Sleeping in 2025: " + str(round(avg_hours_sleeping, 2)))
print("YIPEE!")

Average Hours Sleeping in 2025: 9.03
YIPEE!


Now we will create `avg_activities` which will compute the average time spent on each activity for the population as a whole

In [10]:
avg_activities = duckdb.sql(""" SELECT ROUND(AVG(sleeping), 2) AS avg_sleeping, 
                                       ROUND(AVG(watchingTV), 2) AS avg_tv, 
                                       ROUND(AVG(gaming), 2) AS avg_gaming, 
                                       ROUND(AVG(onComputer), 2) AS avg_onComputer, 
                                       ROUND(AVG(artsCrafts), 2) AS avg_artsCrafts, 
                                       ROUND(AVG(reading), 2) AS avg_reading, 
                                       ROUND(AVG(writing), 2) AS avg_writing, 
                                       ROUND(AVG(exercising), 2) AS avg_exercising 
                                       FROM activities """).df()
print("Average minutes spent on each activity per day (full population):")
avg_activities.head()

Average minutes spent on each activity per day (full population):


,avg_sleeping,avg_tv,avg_gaming,avg_onComputer,avg_artsCrafts,avg_reading,avg_writing,avg_exercising
0,541.73,185.11,20.1,11.37,2.7,20.98,0.35,0.12


Next, we will create `grouped_activities` which is `activities` grouped by the following age categories:
- 15 - 24
- 25 - 34
- 35 - 44
- 45 - 54
- 55 - 64
- 65+

In [7]:
#grouped_activies shows the average minutes spent doing each activity per day for each age group
grouped_activities = duckdb.sql(""" SELECT age_group,
                                           ROUND(AVG(sleeping), 2) AS avg_sleeping,
                                           ROUND(AVG(watchingTV), 2) AS avg_tv,
                                           ROUND(AVG(gaming), 2) AS avg_gaming,
                                           ROUND(AVG(onComputer), 2) AS avg_computer,
                                           ROUND(AVG(artsCrafts), 2) AS avg_artsCrafts,
                                           ROUND(AVG(reading), 2) AS avg_reading,
                                           ROUND(AVG(writing), 2) AS avg_writing,
                                           ROUND(AVG(exercising), 2) AS avg_exercising
                                    FROM (
                                        SELECT
                                            CASE
                                                WHEN age BETWEEN 15 AND 24 THEN '15-24'
                                                WHEN age BETWEEN 25 AND 44 THEN '25-44'
                                                WHEN age BETWEEN 45 AND 64 THEN '45-64'
                                                ELSE '65+'
                                            END AS age_group,
                                            sleeping,
                                            watchingTV,
                                            gaming,
                                            onComputer,
                                            artsCrafts,
                                            reading,
                                            writing,
                                            exercising
                                        FROM activities
                                    )
                                    GROUP BY age_group
                                    ORDER BY age_group """).df()

print("grouped_activities: the average minutes spent on each activity per day for each age group")
grouped_activities.head()

grouped_activities: the average minutes spent on each activity per day for each age group


,age_group,avg_sleeping,avg_tv,avg_gaming,avg_computer,avg_artsCrafts,avg_reading,avg_writing,avg_exercising
0,15-24,610.01,115.74,54.35,19.20,3.72,10.71,0.30,0.14
1,25-44,541.45,119.64,25.13,10.41,1.45,12.75,0.17,0.12
2,45-64,531.52,175.18,10.71,10.08,1.95,15.44,0.36,0.12
3,65+,536.60,261.70,16.94,11.64,4.16,34.61,0.50,0.12


# Limitations:

# Exploratory Data Analysis:

# Questions for Reviewers: